In [ ]:
import numpy as np

In [ ]:


sigmoid= lambda x: 1 / (1 + np.exp(-x))

class BrokenNetwork:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = np.random.randn(input_dim, hidden_dim) / np.sqrt(input_dim)
        self.W2 = np.random.randn(hidden_dim, output_dim) / np.sqrt(hidden_dim)

    def forward(self, X, y):
        self.X = X
        self.y = y

        #print(f"{np.abs(self.W1).mean()=}")
        self.Z1 = X @ self.W1

        self.A1 = 1 / (1 + np.exp(-self.Z1))  # Sigmoid

        self.Z2 = self.A1 @ self.W2

        # Softmax
        exp_z = np.exp(self.Z2)
        #print(f"{exp_z.sum()}")
        self.probs = exp_z / np.sum(exp_z, axis=-1, keepdims=True)
        assert np.allclose(self.probs.sum(axis=-1), 1)

        # Loss
        batch_size = X.shape[0]
        loss = -np.mean(np.log(self.probs[np.arange(batch_size), y]))
        return loss

    def backward(self):
        batch_size = self.X.shape[0]

        # Softmax + CE gradient
        y_onehot = np.zeros_like(self.probs)
        y_onehot[np.arange(batch_size), self.y] = 1
        dZ2 = (self.probs - y_onehot) / batch_size

        # Layer 2
        dW2 = self.A1.T @ dZ2
        dA1 = dZ2 @ self.W2.T
        #print(f"{np.abs(dA1).sum()=}")

        # Through sigmoid
        # grad * q(z) * (1-q(z))
        dZ1 = dA1 * sigmoid(self.Z1) * (1 - sigmoid(self.Z1))
        #print(f"{np.abs(self.Z1).sum()=}")
        #print(f"{np.abs(dZ1).sum()=}")

        # Layer 1
        dW1 = self.X.T @ dZ1
        #print(f"{np.abs(dW1).sum()=}")

        return {'W1': dW1, 'W2': dW2}

def test():
    np.random.seed(42)
    net = BrokenNetwork(5, 10, 3)
    X = np.random.randn(8, 5) * 0.5
    y = np.array([0, 1, 2, 0, 1, 2, 0, 1])

    for i in range(1000):
        loss = net.forward(X, y)
        grads = net.backward()
        net.W1 -= 0.1 * grads['W1']
        net.W2 -= 0.1 * grads['W2']
        if i % 20 == 0:
            print(f"Step {i}: Loss = {loss:.4f}")

test()

Step 0: Loss = 1.1757
Step 20: Loss = 1.0781
Step 40: Loss = 1.0532
Step 60: Loss = 1.0322
Step 80: Loss = 1.0105
Step 100: Loss = 0.9875
Step 120: Loss = 0.9627
Step 140: Loss = 0.9358
Step 160: Loss = 0.9069
Step 180: Loss = 0.8761
Step 200: Loss = 0.8435
Step 220: Loss = 0.8097
Step 240: Loss = 0.7751
Step 260: Loss = 0.7402
Step 280: Loss = 0.7054
Step 300: Loss = 0.6712
Step 320: Loss = 0.6379
Step 340: Loss = 0.6056
Step 360: Loss = 0.5746
Step 380: Loss = 0.5450
Step 400: Loss = 0.5169
Step 420: Loss = 0.4901
Step 440: Loss = 0.4649
Step 460: Loss = 0.4410
Step 480: Loss = 0.4185
Step 500: Loss = 0.3974
Step 520: Loss = 0.3775
Step 540: Loss = 0.3589
Step 560: Loss = 0.3413
Step 580: Loss = 0.3248
Step 600: Loss = 0.3094
Step 620: Loss = 0.2948
Step 640: Loss = 0.2812
Step 660: Loss = 0.2683
Step 680: Loss = 0.2562
Step 700: Loss = 0.2448
Step 720: Loss = 0.2340
Step 740: Loss = 0.2239
Step 760: Loss = 0.2143
Step 780: Loss = 0.2053
Step 800: Loss = 0.1967
Step 820: Loss = 0.188

In [ ]:
import numpy as np

class SingleLayer:
    def __init__(self, input_dim, output_dim):
        self.W = np.random.randn(input_dim, output_dim) * 0.01

    def forward(self, X, Y_target):
        """
        Forward pass:
        1. out = X @ W                    [batch, output_dim]
        2. loss = mean(sum((out - Y_target)^2, axis=1)^0.5)

        This is "mean of L2 norms" - we take L2 norm of each sample's error,
        then average those norms.

        X: [batch, input_dim]
        Y_target: [batch, output_dim]
        """
        self.y = Y_target
        self.b,self.output_dim, self_hidden_dim = X.shape[0], Y_target.shape[1], self.W.shape[1]
        grads = {}
        grads['x'] = X
        out = X @ self.W # [batch, output_dim]
        grads['out1'] = out
        out = (out - Y_target) ** 2 # [batch, output_dim]
        grads['out2'] = out
        out = np.sum(out, axis=1) # [batch]
        grads['out3'] = out
        out = np.sqrt(out) # [batch]
        grads['out4'] = out
        out = np.mean(out) # scalar
        grads['out5'] = out
        self.grads = grads
        return out

    def backward(self):
        """
        Compute gradients.
        Remember: sqrt(x) has derivative 1/(2*sqrt(x))
        """
        grad = np.ones_like(self.grads['out4']) * 1 / self.grads['out4'].size
        print(f"1 {grad=}")
        grad = grad * (1/(2*np.sqrt(self.grads['out3']))) # [batch]
        print(f"2 {grad=}")
        print(f"3 {grad[:, np.newaxis]=}")
        grad = np.ones_like(self.grads['out2']) * grad[:, np.newaxis] # [batch, output_dim]
        grad = grad * 2 * (self.grads['out1'] - self.y) # [batch, output_dim]

        dw = self.grads['x'].T @ grad
        return dw



In [ ]:
layer = SingleLayer(5, 3)
X = np.random.randn(4, 5)  # batch=4, input=5
Y = np.random.randn(4, 3)  # batch=4, output=3
layer.forward(X, Y)
dw = layer.backward()
print(dw)

1 grad=array([0.25, 0.25, 0.25, 0.25])
2 grad=array([0.0995807 , 0.0838243 , 0.05394407, 0.06153225])
3 grad[:, np.newaxis]=array([[0.0995807 ],
       [0.0838243 ],
       [0.05394407],
       [0.06153225]])
[[-0.02713501 -0.10653327 -0.07222854]
 [ 0.08875665  0.3733248   0.14260982]
 [-0.13160877 -0.13753531 -0.2287394 ]
 [-0.09932864 -0.86708233 -0.14886873]
 [ 0.08927678 -0.68069582  0.12893403]]


In [ ]:
import numpy as np

class TwoLayerNet:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.01
        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.01

    def forward(self, X, Y_target):
        """
        This is "mean cubed error" - comparing cubes instead of squares!
        Note: mean over ALL elements (batch and output dimensions)

        X: [batch, input_dim]
        Y_target: [batch, output_dim]

        Forward pass:
        1. H = relu(X @ W1)                     [batch, hidden_dim]
        2. out = H @ W2                         [batch, output_dim]
        3. loss = mean((out^3 - Y_target^3))
        """
        self.output_dim, self.batch = Y_target.shape[1], X.shape[0]
        grads = {}
        grads['Y_target'] = Y_target
        grads['x'] = X
        out = X @ self.W1
        grads['z1'] = out
        out = out * (out > 0)
        grads['a1'] = out
        out = out @ self.W2
        grads['z2'] = out
        self.grads = grads

        out = out ** 3 - Y_target ** 3 # [batch, output_dim]
        out = np.mean(out) # scalar
        return out


    def backward(self):
        """
        Compute gradients for W1 and W2.
        """
        grad = 1
        grad = np.ones_like(self.grads['z1']) * grad / (self.grads['z1'].size)
        grad = grad * (3 * self.grads['z1'] ** 2)

        dw2 = self.grads['a1'].T @ grad
        grad = grad @ self.W2.T
        grad = grad * (self.z1 > 0)

        dw1 = self.grads['x'].T @ grad
        return dw1,dw2






In [ ]:
layer = TwoLayerNet(5, 10, 3)
X = np.random.randn(4, 5)  # batch=4, input=5
Y = np.random.randn(4, 3)  # batch=4, output=3

loss = layer.forward(X, Y)
print(loss)
# grads = layer.backward()
# print(f"Loss: {loss:.4f}")
# print(f"dW1 shape: {grads['dW1'].shape}, dW2 shape: {grads['dW2'].shape}")

-0.029481675345534437


In [ ]:
import numpy as np

class DimensionNet:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.01
        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.01
        self.hidden_dim = hidden_dim


    def forward(self, X, Y_target):
        """
        Forward pass:
        1. Z1 = X @ W1                          [batch, hidden]
        2. H1 = Z1^3                            [batch, hidden]
        3. H1_sum = sum(H1, axis=0)             [hidden]
        4. H1_broadcasted = H1_sum * np.ones((batch, hidden))  [batch, hidden]
        5. out = H1_broadcasted @ W2            [batch, output]
        6. per_sample_loss = sum((out - Y_target)^2, axis=1)  [batch]
        7. loss = mean(per_sample_loss)         scalar

        X: [batch, input_dim]
        Y_target: [batch, output_dim]
        """
        self.dicts = {}
        self.dicts['x'] = X
        self.dicts['y'] = Y_target
        self.b = X.shape[0]

        out = X @ self.W1
        self.dicts['z1'] = out
        out = out ** 3 # 2
        out = out.sum(axis=0) # 3
        out = out * np.ones([self.b, self.hidden_dim]) # 4
        self.dicts['out4'] = out
        out = out @ self.W2 #5
        self.dicts['out5'] = out
        out = (out - Y_target) ** 2 # 6.1 [batch, output]
        self.dicts['out61'] = out
        out = out.sum(axis=1) # 6.2 [batch]
        out = np.mean(out) # 7 scalar
        return out


    def backward(self):
        """
        Compute gradients for W1 and W2.

        Remember:
        - d/dx[x^3] = 3x^2
        - Sum and mean operations expand in backward pass
        """
        grad = 1
        grad = grad * np.ones(self.b) / self.b #7
        print(f"{grad.shape=}")
        grad = grad[:, np.newaxis] * np.ones_like(self.dicts['out61']) # 6.2
        grad = grad * (2 * (self.dicts['out5'] - self.dicts['y'])) # 6.1

        dw2 = self.dicts['out4'].T @ grad

        grad = grad @ self.W2.T # 5 [batch, hidden]
        grad = grad.sum(axis=0) #4 [hidden]
        grad = grad[np.newaxis, :] * np.ones_like(self.dicts['z1'])  #3  [hidden] - > [batch, hidden]

        grad = grad * (3 * self.dicts['z1'] ** 2) # 2

        dw1 = self.dicts['x'].T @ grad

        return {"dW1": dw1, "dW2": dw2}






def test():
    net = DimensionNet(10, 15, 5)
    batch_size = 8

    X = np.random.randn(batch_size, 10) * 0.5
    Y_target = np.random.randn(batch_size, 5) * 0.5

    loss = net.forward(X, Y_target)
    grads = net.backward()

    print(f"Loss: {loss:.4f}")
    print(f"dW1 mean abs: {np.mean(np.abs(grads['dW1'])):.6f}")
    print(f"dW2 mean abs: {np.mean(np.abs(grads['dW2'])):.6f}")

test()

grad.shape=(8,)
Loss: 1.1139
dW1 mean abs: 0.000011
dW2 mean abs: 0.000010


In [ ]:
import numpy as np

sigmoid = lambda x: 1 / (1 + np.exp(-x))

class LinearNet:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.01
        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.01

    def forward(self, X, Y_target):
        """
        Forward pass:
        1. Z1 = X @ W1
        2. H1 = Z1^2 - Z1
        3. H1_avg = mean(H1, axis=0)
        4. H1_broadcast = H1_avg + np.zeros((batch, hidden))
        5. Z2 = H1_broadcast @ W2
        6. out = sigmoid(Z2)
        7. per_sample_loss = mean((out - Y_target)^2, axis=1)
        8. loss = sum(per_sample_loss)

        X: [batch, input_dim]
        Y_target: [batch, output_dim]
        """
        b, hidden_dim = X.shape[0], self.W2.shape[0]
        acts = {}
        acts['b'] = b
        acts['hidden_dim'] = hidden_dim

        acts['x'], acts['y'] = X, Y_target
        out = X @ self.W1
        acts['out4'] = out
        out = out ** 2 - out  # [batch, hidden]
        out = np.mean(out, axis=0) # [hidden]
        out = out + np.zeros([b, hidden_dim]) # [batch, hidden]
        acts['out3'] = out
        out = out @ self.W2
        acts['out2'] = out
        out = sigmoid(out)
        acts['out1'] = out
        out = (out - Y_target) ** 2 # [batch, hidden]
        out = np.mean(out, axis=1) # [batch]
        out = np.sum(out) # scalar


        self.acts = acts
        return out



    def backward(self):
        """
        Compute gradients.

        """
        grad = 1
        grad = np.ones(self.acts['b'])  / self.acts['b']
        grad = 2 * (self.acts['out1'] - self.acts['y']) / self.acts['hidden_dim']

        s = sigmoid(self.acts['out2'])
        grad = grad * (s * (1 - s))

        dw2 = self.acts['out3'].T @ grad
        grad = grad @ self.W2.T # [batch, hidden]
        # [batch, hidden] -> [hidden]
        grad = np.sum(grad, axis=0) # [hidden]
        # [hidden] ->  [batch, hidden]
        print(f"{grad.shape=}")
        grad = grad * grad[np.newaxis, :] * np.ones_like(self.acts['out4']) / self.acts['b']
        print(f"{grad.shape=}")
        # 2 * z1 - 1
        grad = 2 * grad - 1
        #print(f"{grad.shape=}, {self.acts['x'].T.shape=}")
        dw1 = self.acts['x'].T @ grad

        return {'dw1': dw1, 'dw2': dw2}




def test():
    net = LinearNet(10, 15, 5)
    batch_size = 8

    X = np.random.randn(batch_size, 10) * 0.5
    Y_target = np.random.randn(batch_size, 5) * 0.5

    loss = net.forward(X, Y_target)
    grads = net.backward()

    print(f"Loss: {loss:.4f}")
    print(f"dW1 mean abs: {np.mean(np.abs(grads['dw1'])):.6f}")
    print(f"dW2 mean abs: {np.mean(np.abs(grads['dw2'])):.6f}")

test()

grad.shape=(15,)
grad.shape=(8, 15)
Loss: 4.0029
dW1 mean abs: 1.078829
dW2 mean abs: 0.000418


In [ ]:
test()

In [ ]:
import numpy as np

class ProductRuleNet:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.01
        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.01
        self.alpha = np.ones(hidden_dim) * 0.1  # Learnable scaling
        self.hidden_dim = hidden_dim

    def forward(self, X, Y_target):
        """
        Forward pass:
        1. Z1 = X @ W1                           [batch, hidden]
        2. H1 = relu(Z1)                         [batch, hidden]
        3. H1_sum = sum(H1, axis=1, keepdims=True)  [batch, 1]
        4. H1_gated = H1 * H1_sum                [batch, hidden] (H1 times its row sum)
        5. H1_scaled = H1_gated * alpha          [batch, hidden]
        6. H1_final = H1_scaled * exp(-H1_scaled/10)  [batch, hidden] (product rule!)
        7. out = H1_final @ W2                   [batch, output]
        8. loss = mean((out - Y_target)^2)       scalar
        X: [batch, input_dim]
        Y_target: [batch, output_dim]
        """
        acts = {}
        acts['x'], acts['y'] = X, Y_target
        acts['b'] = X.shape[0]
        acts['hidden_dim'] = self.hidden_dim

        out = X @ self.W1
        acts['out4'] = out
        out = (out > 0) * out
        out_sum = np.sum(out, axis=1) # [batch]
        out = out * out_sum[:, np.newaxis] # [batch, hidden]
        acts['out3'] = out
        out = out * self.alpha # [batch, hidden ]
        acts['out2'] = out
        out = out * np.exp( -out / 10) # [batch, hidden ]
        acts['out1'] = out
        out = out @ self.W2
        acts['out0'] = out
        out = (out - Y_target) ** 2
        loss = np.mean(out)
        self.acts = acts
        return loss

    def backward(self):
        acts = self.acts
        grad = 1
        grad = np.ones_like(acts['y']) / acts['y'].size
        grad = grad * 2 * (acts['out0'] - acts['y'])
        dw2 = acts['out1'].T @ grad
        grad = grad @ self.W2.T
        #print(f"{grad.shape}")
        # f(x) * g(x) = f'(x)*g(x) + f(x)*g'(x)
        act_ = acts['out2']
        grad = grad * (np.exp(-act_ / 10) + act_ * -1/10 * np.exp(-act_/10))

        # [batch, hidden] -> [hidden]
        act_ = acts['out3']
        # [hidden]
        dalpha = np.sum(grad * act_, axis=0)
        grad = grad * self.alpha
        # H1_sum: [batch, hidden] -> [batch] ->broadcast-> [batch, hidden]
        # [batch, hidden] -> [batch] -> [batch, hidden]
        grad_sum = np.sum(grad, axis=1)
        # target_shape * grad broadcast
        #print(f"{np.ones([acts['b'], acts['hidden_dim']])=}")
        grad_sum = np.ones([acts['b'], acts['hidden_dim']]) * grad_sum[:, np.newaxis]
        #print(f"{grad.shape}")
        grad = grad + grad_sum

        grad = grad * (acts['out4'] > 0)

        dw1 = acts['x'].T @ grad
        return {'dW1': dw1, 'dW2': dw2, 'dalpha': dalpha}


def test():
    net = ProductRuleNet(10, 15, 5)
    batch_size = 8

    X = np.random.randn(batch_size, 10) * 0.5
    Y_target = np.random.randn(batch_size, 5) * 0.5

    loss = net.forward(X, Y_target)
    grads = net.backward()
    #print(grads)

    print(f"Loss: {loss:.4f}")
    print(f"dW1 mean abs: {np.mean(np.abs(grads['dW1'])):.6f}")
    print(f"dW2 mean abs: {np.mean(np.abs(grads['dW2'])):.6f}")
    print(f"dalpha mean abs: {np.mean(np.abs(grads['dalpha'])):.6f}")

test()

Loss: 0.2432
dW1 mean abs: 0.000110
dW2 mean abs: 0.000013
dalpha mean abs: 0.000002


In [ ]:
np.ones([2,3])

array([[1., 1., 1.],
       [1., 1., 1.]])

In [ ]:
import numpy as np

sigmoid = lambda x: 1 / (1 + np.exp(-x))

class NeuralNet:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.01
        self.W2 = np.random.randn(hidden_dim, hidden_dim) * 0.01
        self.W3 = np.random.randn(hidden_dim, output_dim) * 0.01
        self.alpha = np.ones(hidden_dim) * 0.5
        self.hidden_dim = hidden_dim

    def forward(self, X, Y_target):
        """
        Forward pass:
        1. Z1 = X @ W1
        2. H1 = relu(Z1)
        3. H2 = H1 @ W2
        4. H2_act = H2 * sigmoid(H2) # SiLU
        5. H2_scaled = H2_act * alpha
        6. H3 = H2_scaled + H1
        7. out = H3 @ W3
        8. loss = mean((out - Y_target)^2)

        X: input data
        Y_target: target output
        """
        acts = {}
        acts['x'] = X
        acts['y'] = Y_target
        acts['b'] = X.shape[0]
        acts['hidden_dim'] = self.hidden_dim
        out = X @ self.W1
        acts['h0'] = out
        h1 = (out > 0) * out

        acts['h1'] = h1
        h2 = h1 @ self.W2 #
        acts['h2'] = h2
        h2_act = h2 * sigmoid(h2) #SiLU
        acts['h2_act'] = h2_act
        h2_scaled = h2_act * self.alpha

        h3 = h1 + h2_scaled
        acts['h3'] = h3
        out = h3 @ self.W3 #
        acts['h4'] = out
        out = (out - Y_target) ** 2 #
        out = np.mean(out)
        self.acts = acts
        return out


    def backward(self):
        """
        Compute gradients for all parameters.
        Return dictionary with dW1, dW2, dW3, dalpha.
        """
        acts = self.acts

        grad = 1
        grad = np.ones_like(acts['y']) * grad / acts['y'].size # [batch, output_dim]
        #print(f"{grad=}")
        grad = grad * 2 * (acts['h4'] - acts['y'])
        #print(f"{grad=}")
        dw3 = acts['h3'].T @ grad
        grad = grad @ self.W3.T


        # residual
        # print(f"{np.linalg.norm(acts['h2_act'])=}")
        # print(f"{np.linalg.norm(grad)=}")
        dalpha = np.sum(acts['h2_act'] * grad, 0)
        gres = grad * self.alpha
        # h2 * sigmoid(h2) #SiLU
        # x -> h2
        # f'(x) * g(x) + f(x) * g'(x)
        h2 = acts['h2']
        gres = gres * (sigmoid(h2) + h2 * (sigmoid(h2) * (1 - sigmoid(h2))))

        dw2 = acts['h1'].T @ gres
        gres = gres @ self.W2.T

        grad = grad + gres
        grad = grad * (acts['h0'] > 0)

        dw1 = acts['x'].T @ grad
        return {
            'dW3': dw3,
            'dW1': dw1,
            'dW2': dw2,
            'dalpha': dalpha
        }



def test():
    net = NeuralNet(10, 15, 5)
    batch_size = 8

    X = np.random.randn(batch_size, 10)
    #print(f"{X=}")
    Y_target = np.random.randn(batch_size, 5) * 0.5

    loss = net.forward(X, Y_target)
    print("*"*10)
    grads = net.backward()

    print(f"Loss: {loss:.4f}")
    print(f"dW1 mean abs: {np.mean(np.abs(grads['dW1'])):.6f}")
    print(f"dW2 mean abs: {np.mean(np.abs(grads['dW2'])):.6f}")
    print(f"dW3 mean abs: {np.mean(np.abs(grads['dW3'])):.6f}")
    print(f"dalpha mean abs: {np.mean(np.abs(grads['dalpha'])):.6f}")

test()

**********
Loss: 0.2396
dW1 mean abs: 0.000766
dW2 mean abs: 0.000007
dW3 mean abs: 0.001224
dalpha mean abs: 0.000001


In [ ]:
# np.linalg.norm(acts['h2_act'])=np.float64(0.0022174897906827046)
# np.linalg.norm(grad)=np.float64(0.006857441902937292)

# np.linalg.norm(acts['h2_act'])=np.float64(0.005457083628798975)
# np.linalg.norm(grad)=np.float64(0.0064521655494145565)